In [ ]:
import csv
import os
import sys
sys.path.insert(0, "")

import numpy as np
from sklearn.metrics import accuracy_score

from models.train_model import get_model

from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.banzhaf import exact_banzhaf_index


DATASETS = [
    "../datasets/wine_preprocessed_core.npz",
    "../datasets/yield_preprocessed.npz",
    "../datasets/wine_preprocessed.npz",
    "../datasets/soil_spi_preprocessed.npz",
    "../datasets/house_preprocessed.npz",
    "../datasets/bcw_preprocessed.npz",
]

MODELS               = ["logistic", "svm", "random_forest"]
MAX_FEATURES         = 20
METHODS              = ["shapley", "banzhaf", "gately", "least_core"]
RANDOM_STATE         = 42
OUTPUT_DIR           = "./output_essentiality_check"

os.makedirs(OUTPUT_DIR, exist_ok=True)


class DataObject:
    def __init__(self, n: int, feature_names: list):
        self.data_names = feature_names
        self._n = n

    def __len__(self):
        return self._n


class AccuracyUtility:


    def __init__(self, X_train, X_test, y_train, y_test, model_name,
                 random_state=RANDOM_STATE):
        self.X_train      = X_train
        self.X_test       = X_test
        self.y_train      = y_train
        self.y_test       = y_test
        self.model_name   = model_name
        self.random_state = random_state
        self.n            = X_train.shape[1]
        self.cache        = {}


        majority_class    = np.bincount(y_train.astype(int)).argmax()
        baseline_preds    = np.full_like(y_test, majority_class)
        self.v_baseline   = float(accuracy_score(y_test, baseline_preds))

        self._v_grand     = None

    def _raw_accuracy(self, key):
        if len(key) == 0:
            return self.v_baseline

        X_tr = self.X_train[:, key]
        X_te = self.X_test[:, key]
        model = get_model(self.model_name, random_state=self.random_state)
        try:
            model.fit(X_tr, self.y_train)
            return float(accuracy_score(self.y_test, model.predict(X_te)))
        except Exception as e:
            print(f"  [Utility Error] model={self.model_name}, S={key}, error={e}")
            return self.v_baseline

    def _get_v_grand(self):
        if self._v_grand is None:
            grand_key = tuple(range(self.n))
            self._v_grand = self._raw_accuracy(grand_key)
        return self._v_grand

    def __call__(self, S):
        key = tuple(sorted(int(i) for i in S))

        if key in self.cache:
            return self.cache[key]

        raw     = self._raw_accuracy(key)
        v_N     = self._get_v_grand()
        denom   = v_N - self.v_baseline


        if abs(denom) <= 1e-9:
            print(f"  [WARNING] model={self.model_name} has no predictive power "
                  f"over baseline (v_N={v_N:.4f}, v_baseline={self.v_baseline:.4f}). "
                  f"Normalized utility will be 0.")
            normalized = 0.0
        else:
            normalized = (raw - self.v_baseline) / denom

        self.cache[key] = normalized
        return normalized


class UtilityWrapper:
    def __init__(self, utility: AccuracyUtility, feature_names: list):
        self.util = utility
        self.data = DataObject(utility.n, feature_names)

    def __call__(self, S):
        return self.util(S)


def save_allocations_csv(save_path, feature_names, results, methods):
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["feature_name"] + methods)
        for i, fname in enumerate(feature_names):
            row = [fname]
            for m in methods:
                val = results.get(m)
                row.append(round(float(val[i]), 8) if val is not None else "ERROR")
            writer.writerow(row)
    print(f"    Allocations --> {save_path}")


def save_diagnostics_csv(save_path, feature_names, utility, n_features):

    N   = list(range(n_features))
    v_N = utility(N)
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        writer.writerow(["v_N_normalized (= 1.0 by construction)", round(v_N, 8)])
        writer.writerow(["v_baseline (majority class raw accuracy)", round(utility.v_baseline, 8)])
        writer.writerow(["v_N_raw (grand coalition raw accuracy)", round(utility._get_v_grand(), 8)])
        writer.writerow(["sum_v_singletons_normalized", round(sum(utility([i]) for i in N), 8)])
        writer.writerow([])
        v_N_raw = utility._get_v_grand()
        writer.writerow(["feature_name", "v_singleton_normalized",
                         "v_N_minus_i_normalized", "marginal_normalized", "marginal_raw"])
        for i, fname in enumerate(feature_names):
            loo_cols      = [j for j in N if j != i]
            v_s           = utility([i])
            v_loo_norm    = utility(loo_cols)
            marginal_norm = v_N - v_loo_norm
            v_loo_raw     = utility._raw_accuracy(tuple(sorted(loo_cols)))
            marginal_raw  = v_N_raw - v_loo_raw
            writer.writerow([fname,
                             round(v_s, 8),
                             round(v_loo_norm, 8),
                             round(marginal_norm, 8),
                             round(marginal_raw, 8)])
    print(f"    Diagnostics --> {save_path}")

for dataset_path in DATASETS:
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_path}")
    print(f"{'='*60}")

    data = np.load(dataset_path, allow_pickle=True)
    available_keys = list(data.keys())
    print(f"  Keys in archive: {available_keys}")

    required = ["X_train", "X_test", "y_train", "y_test"]
    if not all(k in available_keys for k in required):
        print(f"  [SKIP] Missing required keys. Expected {required}, found {available_keys}")
        continue

    X_train = data["X_train"]
    X_test  = data["X_test"]
    y_train = data["y_train"]
    y_test  = data["y_test"]
    feature_names = (data["feature_names"].tolist()
                     if "feature_names" in available_keys
                     else [f"f{i}" for i in range(X_train.shape[1])])

    n_features = X_train.shape[1]
    print(f"  X_train={X_train.shape}  X_test={X_test.shape}")
    print(f"  features={feature_names}")

    if n_features > MAX_FEATURES:
        print(f"  [SKIP] {n_features} features > MAX_FEATURES={MAX_FEATURES}")
        continue

    ds_name = os.path.basename(dataset_path).replace(".npz", "")


    DATASET_MODEL_OVERRIDE = {
        "house_preprocessed": ["logistic", "svm"],
    }
    models_for_dataset = DATASET_MODEL_OVERRIDE.get(ds_name, MODELS)
    if models_for_dataset != MODELS:
        print(f"  [INFO] {ds_name}: using models={models_for_dataset} (override)")

    for model_name in models_for_dataset:
        print(f"\n  --- Model: {model_name} ---")

        utility = AccuracyUtility(X_train, X_test, y_train, y_test, model_name)
        u = UtilityWrapper(utility, feature_names)

        results = {}

        for method_name, compute_fn in [
            ("shapley",    lambda fn=exact_shapley_value: fn(u)),
            ("banzhaf",    lambda fn=exact_banzhaf_index: fn(u)),
            ("gately",     lambda fn=exact_gately_value:  fn(u)),
            ("least_core", lambda fn=exact_least_core:    fn(u)),
        ]:
            print(f"    Computing {method_name} ...", end=" ", flush=True)
            try:
                result = compute_fn()
                results[method_name] = result.values if hasattr(result, "values") else result
                print("done")
            except Exception as e:
                print(f"ERROR: {e}")
                results[method_name] = None


        sum_singletons = sum(float(utility([i])) for i in range(n_features))
        is_essential = sum_singletons < 1.0 - 1e-9
        print(f"    Essentiality: sum_v_singletons={sum_singletons:.4f}, "
              f"v(N)=1.0  =>  {'ESSENTIAL' if is_essential else 'NON-ESSENTIAL'}")
        if not is_essential:
            print(f"    [WARN] Non-essential game: Gately may be undefined, "
                  f"allocations less meaningful.")

        alloc_path = os.path.join(OUTPUT_DIR, f"{ds_name}__{model_name}__allocations.csv")
        save_allocations_csv(alloc_path, feature_names, results, METHODS)


        diag_path = os.path.join(OUTPUT_DIR, f"{ds_name}__{model_name}__diagnostics.csv")
        save_diagnostics_csv(diag_path, feature_names, utility, n_features)

print("\nAll done.")


DATASET: ../datasets/wine_preprocessed_core.npz
  Keys in archive: ['X_train', 'X_test', 'y_train', 'y_test', 'feature_names']
  X_train=(142, 13)  X_test=(36, 13)
  features=['Alcohol', 'Malic_acid', 'Ash', 'Alcalinity_of_ash', 'Magnesium', 'Total_phenols', 'Flavanoids', 'Nonflavanoid_phenols', 'Proanthocyanins', 'Color_intensity', 'Hue', 'OD280_OD315', 'Proline']

  --- Model: logistic ---
    Computing shapley ... done
    Computing banzhaf ... done
    Computing gately ... done
    Computing least_core ... done
    Essentiality: sum_v_singletons=4.8095, v(N)=1.0  =>  NON-ESSENTIAL
    [WARN] Non-essential game: Gately may be undefined, allocations less meaningful.
    Allocations --> ./output_essentiality_check/wine_preprocessed_core__logistic__allocations.csv
    Diagnostics --> ./output_essentiality_check/wine_preprocessed_core__logistic__diagnostics.csv

  --- Model: svm ---
    Computing shapley ... done
    Computing banzhaf ... done
    Computing gately ... done
    Computin